# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nCroissant schema @id: {metadata['@id']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all `recordSet` objects and, for each, display its `@id`, fields, and columns. All entity references below are by their `@id` fields.

In [ ]:
# List all record sets and their details
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:\n")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    print(f"  Name: {rs.get('name','')}")
    print(f"  Description: {rs.get('description','')}")
    # List fields and columns by @id if present
    if 'field' in rs:
        print("  Fields and their @id:")
        for f in (rs['field'] if isinstance(rs['field'], list) else [rs['field']]):
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            else:
                print(f"    - {f}")
    if 'column' in rs:
        print("  Columns and their @id:")
        for c in (rs['column'] if isinstance(rs['column'], list) else [rs['column']]):
            if isinstance(c, dict) and '@id' in c:
                print(f"    - {c['@id']}")
            else:
                print(f"    - {c}")
    print()
print(f"All RecordSet @id values: {record_set_ids}")

### Preview records of the main RecordSet

Next, we'll preview the first few records in the main record set. Replace the record set `@id` with the one you wish to explore.

In [ ]:
# Preview first 3 records from the first record set (by @id). Replace with desired record set @id.
if record_set_ids:
    for idx, rec in enumerate(dataset.records(record_set=record_set_ids[0])):
        if idx == 3:
            break
        print(rec)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded RecordSet {rsid}: shape={df.shape}")
    print(f"Columns: {df.columns.tolist()}\n")

# For example, show the first 5 entries/columns of the main record set
main_rs_id = record_set_ids[0]
print(f"Sample records from record set {main_rs_id}:")
display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
1. Select a numeric field by its `@id` (e.g., `cr:field/field_molecular_weight`).
2. Filter records with values above a threshold.
3. Normalize these values.
4. Optionally group by a categorical field (e.g., `cr:field/field_protein_class`).

In [ ]:
# Choose a numeric field (use an actual field @id from previous overview)
# For this dataset, example fields could be 'cr:field/field_molecular_weight', 'cr:field/field_coverage_percentage', etc.
main_df = dataframes[main_rs_id]

# Try to automatically pick a likely numeric field
import re
numeric_candidates = [col for col in main_df.columns if re.search(r'(weight|count|coverage|abundance|MW)', col, re.I)]
numeric_field = numeric_candidates[0] if numeric_candidates else main_df.columns[1]
print(f"Using numeric field for demo: {numeric_field}\n")

# Ensure numeric conversion
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
threshold = main_df[numeric_field].quantile(0.8)  # use 80th percentile
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 20%): {len(filtered_df)} records\n")
display(filtered_df.head())

# Normalize the chosen field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a likely grouping field
group_candidates = [col for col in filtered_df.columns if re.search(r'(type|class|state|group|sample)', col, re.I)]
group_field = group_candidates[0] if group_candidates else None
if group_field is not None:
    print(f"Grouping by: {group_field}")
    grouped = filtered_df.groupby(group_field)[numeric_field].mean()
    print("Mean by group:")
    display(grouped)
else:
    print("No grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If group field exists, boxplot
if group_field is not None:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=main_df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using `mlcroissant`, loaded record sets by their `@id`, selected fields for analysis, filtered and normalized numeric data, and visualized important distributions.

Further exploration can include deeper domain-specific analyses, correlation studies, or advanced machine learning techniques using the processed DataFrame(s).